In [1]:
import random
import torch
from rdkit import Chem
from rdkit.Chem import BRICS
from torch_geometric.data import Data
from tqdm import tqdm
import pandas as pd
from polygraphpy.gnn.pre_processing import PreProcess

# Setup
preprocess = PreProcess(
    input_csv='polarizability_data_monomer.csv',
    train_input_data_path='../polygraphpy/data/training_input_data/',
    polymer_type='monomer',
    target='static_polarizability',
    gnn_output_path='./'
)

df = preprocess.run()
atoms_list, bonds_list = preprocess.extract_atoms_and_bonds_features_from_monomer_smiles()
atom_encoder = preprocess.make_encoder(pd.DataFrame(atoms_list).drop_duplicates().reset_index(drop=True))
bond_encoder = preprocess.make_encoder(pd.DataFrame(bonds_list).drop_duplicates().reset_index(drop=True))

model = torch.load('../polygraphpy/data/gnn_output/model_gcn.pt', weights_only=False)
print(model)

/home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading GNN input file.
Removing outliers...
Making data standardization...
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:03<00:00, 2749.55it/s]


Making feature encoder.
Making feature encoder.
Training data preparation starting. 9003 to go.


9003it [00:40, 221.94it/s]


Training data preparation finished.
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:03<00:00, 2850.09it/s]


Making feature encoder.
Making feature encoder.
GCN(
  (conv1): GCNConv(75, 225)
  (conv2): GCNConv(225, 225)
  (conv3): GCNConv(225, 225)
  (lin1): Linear(in_features=225, out_features=225, bias=True)
  (lin2): Linear(in_features=225, out_features=225, bias=True)
  (lin3): Linear(in_features=225, out_features=225, bias=True)
  (output): Linear(in_features=225, out_features=1, bias=True)
)


In [2]:
import numpy as np

class FragmentGA:
    def __init__(self, csv_path, model, preprocess: PreProcess, atom_encoder, bond_encoder, population_size=20):
        self.df = pd.read_csv(csv_path)
        self.model = model.eval()
        self.preprocess = preprocess
        self.atom_encoder = atom_encoder
        self.bond_encoder = bond_encoder
        self.population_size = population_size

        self.fragments = self._extract_fragments()

        print("Fragments sample: ")
        print(self.fragments[:25])
        self.device = next(model.parameters()).device

    def _extract_fragments(self):
        all_frags = set()
        for smi in self.df['smiles']:
            mol = Chem.MolFromSmiles(smi, sanitize=True)
            if mol is None:
                continue
            try:
                Chem.RemoveStereochemistry(mol)  # Remove stereochemistry
                frags = BRICS.BRICSDecompose(mol, minFragmentSize=3)
                valid_frags = {f for f in frags if Chem.MolFromSmiles(f, sanitize=True) is not None}
                all_frags.update(valid_frags)
            except:
                continue
        fragments = list(all_frags)
        print(f"Extracted {len(fragments)} valid fragments")
        return fragments if fragments else self.fragments

    def _build_random_molecule(self):
        if not self.fragments:
            print("No fragments available.")
            return None
        for _ in range(1000):
            frags = random.sample(self.fragments, k=random.randint(2, 6))
            try:
                mol_frags = [Chem.MolFromSmiles(f, sanitize=True) for f in frags]
                if None in mol_frags:
                    continue
                new_mol = BRICS.BRICSBuild(mol_frags)
                for mol in new_mol:
                    Chem.RemoveStereochemistry(mol)
                    smi = Chem.MolToSmiles(mol, isomericSmiles=False)
                    mol = Chem.MolFromSmiles(smi, sanitize=True)
                    if mol:
                        Chem.SanitizeMol(mol)
                        return smi
            except Exception as e:
                print(f"Failed to build molecule with fragments {frags}: {str(e)}")
                continue
        print("All attempts to build molecule failed.")
        return None
    
    def _mol_to_data(self, smiles):
        try:
            atoms = []
            bonds = []
            m1 = Chem.MolFromSmiles(smiles, sanitize=True)
            if m1 is None:
                print(f"Invalid SMILES: {smiles}")
                return None
            m1 = Chem.AddHs(m1)
            
            atoms = self.preprocess.get_nodes_information(m1, [], chain_size=0)
            if not atoms:
                print(f"No atoms extracted for SMILES: {smiles}")
                return None
            df_nodes = pd.DataFrame(atoms)
            nodes_features = pd.DataFrame(self.atom_encoder.transform(df_nodes.drop(['idx'], axis=1)).toarray())
            zero_vector = np.zeros((nodes_features.shape[0], 1))
            nodes_features = pd.concat([nodes_features, pd.DataFrame(zero_vector)], axis=1)
            nodes_features = pd.concat([nodes_features, pd.DataFrame(zero_vector)], axis=1)
            x = torch.tensor(nodes_features.astype('float32').values)
            
            bonds = self.preprocess.get_bonds_information(m1, [])
            if not bonds:
                print(f"No bonds extracted for SMILES: {smiles}")
                return None
            df_bonds = pd.DataFrame(bonds)
            edge_index = torch.tensor([
                df_bonds.begin_idx.to_list() + df_bonds.end_idx.to_list(),
                df_bonds.end_idx.to_list() + df_bonds.begin_idx.to_list()
            ])
            
            edge_attrs = df_bonds[['type', 'is_conjugated', 'is_aromatic']]
            edge_attrs = pd.concat([edge_attrs, edge_attrs.sort_index(ascending=False)])
            edge_attr = torch.tensor(self.bond_encoder.transform(edge_attrs).toarray(), dtype=torch.float32)
            
            edge_weight = torch.tensor([1.0] * edge_index.shape[1], dtype=torch.float32)
            
            mol_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, edge_weight=edge_weight)
            mol_data.validate()
            return mol_data
        except Exception as e:
            print(f"Error in _mol_to_data for SMILES {smiles}: {str(e)}")
            return None
    
    def _evaluate_fitness(self, smiles, target_polarizability):
        try:
            data = self._mol_to_data(smiles)
            if data is None:
                return -1.0
            data = data.to(self.device)
            with torch.no_grad():
                prediction = self.model(data.x, data.edge_index, data.edge_weight, torch.tensor(np.array([0])).to(self.device)).item()
            score = -abs(prediction - target_polarizability)
            return score if prediction > 0 else -1.0  # Ensure positive predictions
        except Exception as e:
            print(f"Error evaluating fitness for SMILES {smiles}: {str(e)}")
            return -1.0

    def run(self, generations=10, target_polarizability=500.0):
        population = [self._build_random_molecule() for _ in range(self.population_size)]
        population = [p for p in population if p is not None]
        if not population:
            print("Initial population empty. Check fragment generation.")
            return []

        for gen in tqdm(range(generations)):
            print(f"Generation {gen + 1}")
            fitness_scores = [
                (smi, self._evaluate_fitness(smi, target_polarizability))
                for smi in tqdm(population)
            ]

            # Selection
            fitness_scores.sort(key=lambda x: x[1], reverse=True)
            top_individuals = fitness_scores[:self.population_size // 2]
            if len(top_individuals) == 0:
                print("No valid molecules in this generation. Reinitializing population...")
                population = [self._build_random_molecule() for _ in range(self.population_size)]
                population = [p for p in population if p is not None]
                continue

            # Crossover
            top_frags = set()
            for smi, _ in top_individuals:
                mol = Chem.MolFromSmiles(smi, sanitize=True)
                if mol is None:
                    continue
                try:
                    top_frags.update(BRICS.BRICSDecompose(mol, minFragmentSize=3))
                except:
                    continue
            original_fragments = self.fragments
            self.fragments = list(top_frags) if top_frags else original_fragments
            new_population = []
            for _ in range(self.population_size):
                new_smi = self._build_random_molecule()
                if new_smi:
                    new_population.append(new_smi)
            self.fragments = original_fragments

            # Fallback if crossover fails
            population = new_population
            if not population:
                print("Crossover failed to produce valid molecules. Reinitializing population...")
                population = [self._build_random_molecule() for _ in range(self.population_size)]
                population = [p for p in population if p is not None]

        return fitness_scores

In [3]:
ga = FragmentGA(csv_path='polarizability_data_monomer.csv',
                model=model,
                preprocess=preprocess,
                atom_encoder=atom_encoder,
                bond_encoder=bond_encoder,
                population_size=80)

target_value = 0.555
top_molecules = ga.run(generations=10, target_polarizability=target_value)

# Top 5 candidates
for smi, score in sorted(top_molecules, key=lambda x: x[1], reverse=True)[:5]:
    print(f"SMILES: {smi} | Fitness: {score:.4f}")

Extracted 6398 valid fragments
Fragments sample: 
['[7*]Cc1c(C)cc(OC)cc1OC', '[3*]Oc1c(F)c(F)c(F)c(F)c1F', '[5*]NC=C(c1ccc(C)cc1)c1c(C[7*])n(C)c2ccccc12', '[3*]OC(=C)C(=O)OC', '[5*]NC=C(C(=O)OC)S(=O)(=O)c1ccc(Cl)cc1', '[5*]Nc1ccc(Cl)c(F)c1', '[5*]Nc1ccc(Cl)c(C[7*])c1F', '[16*]c1ccc(Cl)cc1Br', '[7*]Cc1cc([N+](=O)[O-])ccc1F', '[1*]C(=O)c1ccc([16*])cc1C[7*]', '[16*]c1ccc(I)cc1', '[9*]n1cc(Cl)c2ccccc21', '[9*]n1nc(C)c2c(N)ncnc21', '[4*]CCc1c([16*])c(C)c([14*])n1C=C', '[7*]Cc1ccccc1I', '[6*]C(=O)c1ccc(F)cc1', '[16*]c1cc(F)c(Cl)cc1Cl', '[7*]Cc1ccc(C#N)o1', '[4*]CCc1cn(C)c2c(C[7*])ccc(OC)c12', '[16*]c1ccc(O)c(OC)c1', '[13*]C1OC(CO)C(O)C(O)C1O', '[1*]C(=O)C(=C)C(=O)OC', '[1*]C(=O)C([7*])C=NC(C)CO', '[1*]C(=O)C([7*])C', '[15*]C1CC1']


  0%|          | 0/10 [00:00<?, ?it/s]

Generation 1


 10%|█         | 1/10 [00:01<00:13,  1.49s/it]

Generation 2


...

Generation 9


 90%|█████████ | 9/10 [02:33<00:28, 28.76s/it]

Generation 10


100%|██████████| 10/10 [02:48<00:00, 16.85s/it]

SMILES: N#Cc1ccc(Oc2c3ccccc3cc3ccccc23)c(Cl)c1 | Fitness: -0.0012
SMILES: N#Cc1ccc(OC2SCC3NC(=O)NC32)c(Cl)c1 | Fitness: -0.0023
SMILES: N#CC1(c2ccccc2)C(C=CC2=C(N)c3ccccc3C2(C#N)c2ccccc2)=C(N)c2ccccc21 | Fitness: -0.0038
SMILES: N#Cc1ccc(Oc2cc(Oc3ccnc(Oc4ccc(Cl)cc4OC4SCC5NC(=O)NC54)c3)ccn2)c(Cl)c1 | Fitness: -0.0052
SMILES: O=[N+]([O-])c1cc(COc2ccc(Cl)cc2OCc2cc([N+](=O)[O-])cc([N+](=O)[O-])c2)cc([N+](=O)[O-])c1 | Fitness: -0.0082
